# Document Loader 실습

PDF 같은 외부 파일을 LangChain의 `Document` 객체로 불러오는 방법을 실습한다. 먼저 `Document` 객체 자체의 구조를 살펴본 뒤, 실제 PDF 파일을 여러 방식으로 불러오고(loader), 불러온 문서를 작은 조각(chunk)으로 나눠보는 것(splitter)까지 다룬다.

In [2]:
from langchain_core.documents import Document

# page_content: 문서의 실제 텍스트 내용을 담아 Document 객체를 생성.
document = Document(page_content="안녕하세요? 이건 랭체인의 다큐먼트입니다.")

## 1. Document 객체의 구조

`Document`는 텍스트(`page_content`)와 부가정보(`metadata`)를 함께 담는 단순한 자료구조다. `__dict__`로 내부 속성을 보면 `id`(아직 지정 안 해서 `None`), `metadata`(빈 dict), `page_content`, `type`(`"Document"` 고정값) 네 가지로 이루어져 있는 것을 알 수 있다.

In [3]:
document.__dict__

{'id': None,
 'metadata': {},
 'page_content': '안녕하세요? 이건 랭체인의 다큐먼트입니다.',
 'type': 'Document'}

## 2. metadata 채워넣기

`metadata`는 평범한 파이썬 `dict`라서, 키를 추가하는 것만으로 원하는 부가정보(출처, 페이지 번호, 작성자 등)를 자유롭게 붙일 수 있다.

In [4]:
document.metadata["source"] = "TeddyNote"
document.metadata["page"] = 1
document.metadata["author"] = "Teddy"

추가한 메타데이터를 확인한다.

In [5]:
document.metadata

{'source': 'TeddyNote', 'page': 1, 'author': 'Teddy'}

## 3. 실제 PDF 파일 준비

`data` 폴더에 있는 PDF 파일("SPRI_AI_Brief_2023년12월호_F.pdf", 소프트웨어정책연구소의 AI 브리프 보고서) 경로를 지정한다. 이 파일을 여러 방식의 로더로 불러오면서 비교해본다.

In [6]:
FILE_PATH = "./data/SPRI_AI_Brief_2023년12월호_F.pdf"

## 4. PyPDFLoader로 PDF 불러오기

`PyPDFLoader`는 PDF를 페이지 단위로 읽어서 각 페이지를 하나의 `Document`로 만들어주는 가장 기본적인 PDF 로더다. `langchain_community` 소속이라 "sunset(지원 종료 예정)" 경고가 뜨는데, `langchain-community` 패키지 자체가 점차 별도 통합 패키지들로 옮겨가고 있어서 나오는 경고이고 기능은 정상 동작한다.

In [7]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(FILE_PATH)

C:\Users\user\AppData\Local\Temp\ipykernel_12584\3390251491.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


`loader.load()`를 호출하면 실제로 파일을 읽어서 `Document` 리스트를 반환한다. PDF가 23페이지라서 `Document`도 23개(페이지당 1개)가 만들어진다.

In [8]:
docs = loader.load()

len(docs)

23

## 5. 개별 문서(페이지) 살펴보기

`docs[5]`(6번째 페이지)를 보면, `metadata`에 PDF 자체의 메타정보(`producer`, `creator`, `creationdate`, `author`, `total_pages`, `page`, `page_label` 등)가 자동으로 채워져 있고, `page_content`에는 그 페이지의 텍스트가 그대로 들어있다.

In [9]:
docs[5]

Document(metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 5, 'page_label': '6'}, page_content='1. 정책/법제  2. 기업/산업 3. 기술/연구  4. 인력/교육\n영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언\nn 영국 블레츨리 파크에서 개최된 AI 안전성 정상회의에 참가한 28개국들이 AI 안전 보장을 \n위한 협력 방안을 담은 블레츨리 선언을 발표\nn 첨단 AI를 개발하는 국가와 기업들은 AI 시스템에 대한 안전 테스트 계획에 합의했으며, \n영국의 AI 안전 연구소가 전 세계 국가와 협력해 테스트를 주도할 예정 \nKEY Contents\n£ AI 안전성 정상회의 참가국들, 블레츨리 선언 통해 AI 안전 보장을 위한 협력에 합의\nn 2023년 11월 1~2일 영국 블레츨리 파크에서 열린 AI 안전성 정상회의(AI Safety Summit)에 \n참가한 28개국 대표들이 AI 위험 관리를 위한 ‘블레츨리 선언’을 발표 \n∙ 선언은 AI 안전 보장을 위해 국가, 국제기구, 기업, 시민사회, 학계를 포함한 모든 이해관계자의 협력이 \n중요하다고 강조했으며, 특히 최첨단 AI 시스템 개발 기업은 안전 평가를 비롯한 적절한 조치를 취하여 \nAI 시스템의 안전을 보장할 책임이 있다고 지적\n∙ 각국은 AI 안전 보장을 위해 첨단 AI 개발기업의 투명성 향상, 적절한 평가지표와 안전 테스트 도구 \n개발, 공공부문 역량 구축과 과학 연구개발 등의 분

## 6. 다른 로더와 비교: PyMuPDF4LLMLoader

`langchain_pymupdf4llm`의 `PyMuPDF4LLMLoader`는 PDF를 LLM이 다루기 좋은 마크다운 형태(제목에 `#`, 굵은 글씨에 `**` 등)로 변환해서 추출해주는 로더다. `use_layout=False`는 원본의 표/레이아웃을 그대로 재현하기보다 읽기 편한 마크다운 텍스트로 단순화해서 추출하라는 옵션이다. 페이지 수(23)는 동일하지만, 출력된 텍스트 형식이 `PyPDFLoader`와는 다른 것을 확인할 수 있다.

In [10]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

future_loader = PyMuPDF4LLMLoader(FILE_PATH, use_layout=False)
future_docs = future_loader.load()

print(len(future_docs))
print(future_docs[5].page_content[:200])

23
**1. 정책/법제** 2. 기업/산업 3. 기술/연구 4. 인력/교육

###### 영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언


**KEY Contents**


n 영국 블레츨리 파크에서 개최된 AI 안전성 정상회의에 참가한 28개국들이 AI 안전 보장을

위한 협력 방안을 담은 블레츨리 선언을 발표


n 첨단 AI를


## 7. PyPDFLoader의 레이아웃 보존 모드

`PyPDFLoader(FILE_PATH, extraction_mode="layout")`로 불러오면, 원본 PDF에 보이는 것과 비슷하게 띄어쓰기·정렬 등 시각적 레이아웃을 최대한 살려서 텍스트를 추출한다. 기본 모드보다 원본의 표/문단 구조를 짐작하기엔 좋지만, 그만큼 공백이 많이 섞여서 텍스트가 다소 지저분해 보일 수 있다.

In [11]:
layout_loader = PyPDFLoader(FILE_PATH, extraction_mode="layout")
layout_docs = layout_loader.load()

print(layout_docs[5].page_content[:300])

1. 정책/법제             2. 기업/산업             3. 기술/연구             4. 인력/교육

영국 AI 안전성      정상회의에       참가한 28개국, AI 위험에 공동 대응 선언

      KEY Contents

   n  영국   블레츨리    파크에서    개최된   AI 안전성   정상회의에     참가한   28개국들이    AI 안전   보장을
      위한   협력  방안을   담은   블레츨리    선언을   발표

   n  첨단   AI를 개발하는     국가와  


## 8. 문서를 작은 조각(chunk)으로 나누기

`RecursiveCharacterTextSplitter`는 긴 문서를 지정한 글자 수 단위로 잘라서 여러 개의 작은 `Document`로 쪼개준다. 나중에 RAG(검색 기반 답변) 등에서 문서를 다룰 때, 문서 전체를 통째로 쓰기보다 이렇게 작은 조각 단위로 다루는 경우가 많다.

- `chunk_size=200` : 조각 하나의 최대 글자 수
- `chunk_overlap=0` : 조각들 사이에 겹치는 부분 없음

23개였던 문서(페이지)가 200자 단위로 쪼개지면서 173개의 조각으로 늘어난 것을 확인할 수 있다.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)

loader = PyPDFLoader(FILE_PATH)
docs = loader.load()

split_docs = text_splitter.split_documents(docs)

print(f"문서의 길이: {len(split_docs)}")

split_docs[10]

문서의 길이: 173


Document(metadata={'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2'}, page_content='Ⅱ. 주요 행사\n   ▹CES 2024 ·····························································································································19')

## 9. lazy_load(): 지연 로딩

`load()`는 호출하는 즉시 전체 페이지를 한 번에 다 읽어서 리스트로 반환한다. 반면 `lazy_load()`는 호출해도 아직 아무 페이지도 읽지 않고, **제너레이터(generator) 객체만** 반환한다. 실제로 값을 하나씩 꺼내 쓸 때(순회할 때)마다 그제서야 페이지를 하나씩 읽어온다. 문서가 아주 많아서 한 번에 메모리에 다 올리기 부담스러울 때 유용하다.

In [13]:
loader.lazy_load()  # 아직 아무 것도 읽지 않은 상태의 제너레이터 객체

<generator object PyPDFLoader.lazy_load at 0x000001A35B14D460>

`for`문으로 순회하면 그제서야 한 페이지씩 실제로 읽어오면서 처리한다.

In [14]:
for doc in loader.lazy_load():
    # 순회하는 시점에 한 페이지씩 실제로 로드된다.
    print(doc.metadata)

{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2'}
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 2, 'page_label': '3'}
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 

## 10. 비동기(async) 버전: aload() / alazy_load()

`aload()`는 `load()`의 비동기 버전이다. `await`로 호출하며, `load()`와 마찬가지로 전체 페이지를 한 번에 리스트로 반환한다. (주피터 노트북은 셀에서 바로 `await`를 쓸 수 있도록 top-level await를 지원한다.)

In [15]:
adocs = await loader.aload()
len(adocs)

23

`alazy_load()`는 `lazy_load()`의 비동기 버전이다. `async for`로 순회하면서 한 페이지씩 비동기로 읽어온다.

In [16]:
async for doc in loader.alazy_load():
    print(doc.metadata)

{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2'}
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 'author': 'dj', 'moddate': '2023-12-08T13:28:38+09:00', 'pdfversion': '1.4', 'source': './data/SPRI_AI_Brief_2023년12월호_F.pdf', 'total_pages': 23, 'page': 2, 'page_label': '3'}
{'producer': 'Hancom PDF 1.3.0.542', 'creator': 'Hwp 2018 10.0.0.13462', 'creationdate': '2023-12-08T13:28:38+09:00', 

## 정리

| 메서드 | 반환 방식 | 동기/비동기 |
|---|---|---|
| `load()` | 전체를 한 번에 리스트로 | 동기 |
| `lazy_load()` | 순회할 때마다 하나씩(제너레이터) | 동기 |
| `aload()` | 전체를 한 번에 리스트로 | 비동기 (`await`) |
| `alazy_load()` | 순회할 때마다 하나씩(비동기 제너레이터) | 비동기 (`async for`) |

로더 자체도 `PyPDFLoader`(기본/레이아웃 보존 모드), `PyMuPDF4LLMLoader`(마크다운 형태로 추출)처럼 여러 종류가 있어서, 이후 처리(요약, 검색 등) 목적에 맞는 형식으로 불러올 수 있다. 불러온 문서는 `RecursiveCharacterTextSplitter`로 원하는 크기의 조각으로 잘라서 활용한다.